# 10.2 - Designing Tool Schemas That Actually Work

**Module 10 - Tool Use & MCP** | Netsetos GenAI Engineering

This notebook builds the schema-engineering toolkit for tool-calling LLMs: one Pydantic model that emits a schema for every provider, description routing, the cross-provider strict-mode matrix, type/enum/nesting rules, tool-count granularity, side-effect safety, versioning with golden tests, and Indic-correct validation. Every concept is a tiny runnable harness you can read as plain Python - no framework magic.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install (optionally) the provider SDKs and note that the lesson uses fixed, seeded example values so runs are reproducible. Nothing here calls a model - it just prepares the environment for the runnable cells that follow.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai -q google-genai

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A commented-out `pip install` line plus a reproducibility note - pure environment prep, no executable logic that produces output.

**How the code works - step by step**
- **`pip install` line (commented)** - uncomment to install `anthropic`, `openai`, and `google-genai` if you are on Colab or a fresh machine; most later cells are pure-Python and need none of them.
- **Reproducibility comment** - the lesson's example values are seeded/fixed so every run prints the same numbers.

**In one line:** optional SDK install, otherwise a no-op setup cell.

**What you'll see:** (no output - environment prep)

## 1 - Pydantic: one model, every provider

Instead of hand-writing JSON Schema per provider, define your tool's inputs once as a Pydantic `BaseModel` and get three things free: runtime validation, self-documentation, and a JSON Schema (dialect 2020-12) you adapt into each provider's tool format. This is the single-source-of-truth pattern that stops your OpenAI, Anthropic, Gemini, and MCP schemas from drifting apart.

In [ ]:
# ONE PYDANTIC MODEL -> validation + docs + JSON Schema for EVERY provider.
from pydantic import BaseModel, Field
from typing import Optional, Literal
import json

class SearchCourses(BaseModel):
    """Search the Netsetos course catalog by topic and price."""
    topic: Literal["genai", "python", "cloud", "data"] = Field(description="Course topic to search")
    max_price_inr: Optional[int] = Field(default=None, ge=0, le=100000,
                                         description="Max price in INR; no cap if omitted")
    sort_by: Literal["price", "rating", "duration"] = Field(default="rating", description="Sort order")

core = SearchCourses.model_json_schema()          # JSON Schema 2020-12, generated for free
print("properties:", list(core["properties"].keys()))
print("author-required:", core.get("required"))   # only fields with NO default

def to_openai(name, desc, schema):                 # STRICT subset: all-required + additionalProperties:false
    s = json.loads(json.dumps(schema)); s["additionalProperties"] = False
    s["required"] = list(s.get("properties", {}).keys())    # strict: EVERY property required
    return {"type": "function", "function": {"name": name, "description": desc, "strict": True, "parameters": s}}
def to_anthropic(name, desc, schema): return {"name": name, "description": desc, "input_schema": schema}
def to_mcp(name, desc, schema):       return {"name": name, "description": desc, "inputSchema": schema}  # full 2020-12
# gemini (google-genai): you pass the PYTHON FUNCTION; the SDK builds the declaration from hints+docstring

oa = to_openai("search_courses", "Search the catalog", core)
print("OpenAI  parameters.required (strict):", oa["function"]["parameters"]["required"])
print("Anthropic key:", list(to_anthropic("x", "", core).keys())[-1],
      "| MCP key:", list(to_mcp("x", "", core).keys())[-1])

print("validate GOOD:", SearchCourses(topic="genai", max_price_inr=15000).model_dump())
try:
    SearchCourses(topic="genai", sort_by="popularity")     # not in the Literal
except Exception as e:
    print("validate BAD:", str(e).splitlines()[0])

# Output:
# properties: ['topic', 'max_price_inr', 'sort_by']
# author-required: ['topic']
# OpenAI  parameters.required (strict): ['topic', 'max_price_inr', 'sort_by']
# Anthropic key: input_schema | MCP key: inputSchema
# validate GOOD: {'topic': 'genai', 'max_price_inr': 15000, 'sort_by': 'rating'}
# validate BAD: 1 validation error for SearchCourses

**Explanation**

The cell defines one `SearchCourses` model, generates its JSON Schema, then runs it through four tiny adapter functions - one per provider - and finally proves the validation is real. Read it as: author the model once, then *export* (not rewrite) a schema shaped for each target.

**How the code works - step by step**
- **`SearchCourses(BaseModel)`** - one topic `Literal` (becomes an enum), an `Optional[int]` price with `ge/le` bounds (a field with a default = optional), and a `sort_by` `Literal` with a default.
- **`model_json_schema()`** - emits JSON Schema 2020-12 for free; its `required` list contains only `topic`, the one field with no default.
- **`to_openai`** - rewrites `required` to EVERY property and sets `additionalProperties:false` - the strict contract; also flips `strict:True`.
- **`to_anthropic` / `to_mcp`** - wrap the same core under `input_schema` vs `inputSchema` (the keys differ); Gemini instead takes the raw Python function.
- **validation block** - a GOOD instance dumps clean; a BAD one (`sort_by="popularity"`, not in the `Literal`) raises a validation error - the same rule the model must obey.

**In one line:** one BaseModel -> validation + docs + a JSON Schema you adapt per provider.

**What you'll see:** Prints the property list `['topic','max_price_inr','sort_by']`, the author-required `['topic']`, then the OpenAI-strict required expanded to all three, the Anthropic/MCP key names (`input_schema` / `inputSchema`), a successful `model_dump`, and finally `validate BAD: 1 validation error for SearchCourses` for the out-of-enum value.

## 2 - Descriptions are prompts

The model reads a tool's description exactly like a prompt to decide WHEN to call it, so the description is the highest-ROI lever in tool design. A good one has four parts - capability, WHEN to use, WHAT it returns, and (the part everyone skips) NEGATIVE examples. This cell proves the shape of the fix with a keyword router standing in for the LLM.

In [ ]:
# DESCRIPTIONS ARE PROMPTS: a tiny keyword router shows vague vs precise descriptions.
def route(query, tools):
    """Pick the best tool by keyword overlap; a NEGATIVE keyword vetoes a tool."""
    q = query.lower(); best, best_score = "answer-directly", 0
    for name, keywords, negatives in tools:
        if any(n in q for n in negatives):        # "Do NOT use for..." blocks a wrong match
            continue
        score = sum(1 for k in keywords if k in q)
        if score > best_score:
            best, best_score = name, score
    return best

VAGUE = [("search", ["course", "weather", "data", "info"], []),      # overlapping, no WHEN, no negatives
         ("book",   ["course", "weather", "data"], [])]
PRECISE = [
    ("search_courses", ["course", "price", "cost", "catalog", "which", "list"], ["book", "enroll", "forecast"]),
    ("book_course",    ["book", "enroll", "buy", "purchase"], ["search", "which", "list", "cost"]),
    ("get_weather",    ["weather", "temperature"], ["forecast", "history", "climate", "air quality"]),
]
qs = ["Which GenAI courses do you have and their price?", "Book me the cloud course",
      "What is the weather in Pune?", "What is the air quality in Delhi?"]
print("VAGUE descriptions:")
for q in qs: print(f"  {q[:38]:38s} -> {route(q, VAGUE)}")
print("PRECISE descriptions (capability + WHEN + negatives):")
for q in qs: print(f"  {q[:38]:38s} -> {route(q, PRECISE)}")

# Output:
# VAGUE descriptions:
#   Which GenAI courses do you have and th -> search
#   Book me the cloud course               -> search    <- WRONG (should book)
#   What is the weather in Pune?           -> search    <- WRONG (should be weather)
#   What is the air quality in Delhi?      -> answer-directly
# PRECISE descriptions (capability + WHEN + negatives):
#   Which GenAI courses do you have and th -> search_courses
#   Book me the cloud course               -> book_course
#   What is the weather in Pune?           -> get_weather
#   What is the air quality in Delhi?      -> answer-directly   <- negatives blocked get_weather

**Explanation**

A toy `route` function scores tools by keyword overlap and lets a NEGATIVE keyword veto a match, then runs the same four queries against a VAGUE tool set and a PRECISE one. It is a stand-in for the model's routing decision - the point is the SHAPE of the improvement, not the router itself.

**How the code works - step by step**
- **`route(query, tools)`** - lowercases the query, skips any tool whose negative keyword appears, then picks the tool with the highest keyword overlap (defaulting to `answer-directly`).
- **`VAGUE`** - tools with overlapping keywords, no WHEN signal, and empty negative lists - so queries collide.
- **`PRECISE`** - each tool gets distinct capability keywords plus negatives (`get_weather` vetoes `air quality`, `climate`, etc.).
- **query loop** - runs four sample queries through each tool set and prints the routing.

**In one line:** capability + WHEN + NEGATIVE keywords turn mis-routes into correct routes.

**What you'll see:** Under VAGUE, "Book me..." and "weather..." both wrongly route to `search` (flagged `<- WRONG`). Under PRECISE, each query routes correctly, and "air quality in Delhi" falls through to `answer-directly` because the negatives blocked `get_weather`.

## 3 - The real routing test (live model)

The keyword router was a stand-in; this cell confirms the same routing with an actual model via the current `google-genai` SDK and automatic function calling. It is illustrative and minimal - production would wrap the call in try/except with a timeout.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY` or `GEMINI_API_KEY`).

In [ ]:
# THE REAL ROUTING TEST (illustrative; needs GOOGLE_API_KEY). CURRENT google-genai SDK.
# Minimal example only - production wraps each call in try/except with a timeout=.
import os
from google import genai
from google.genai import types

def search_courses(topic: str, max_price_inr: int = 0) -> dict:
    """Search the Netsetos catalog by topic and price. Use when the user asks WHICH
    courses exist, their price, or details. Do NOT use to enroll or book a course."""
    return {"results": []}

if os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"):
    client = genai.Client()                                    # GOOGLE_API_KEY
    chat = client.chats.create(model="gemini-2.5-flash",
        config=types.GenerateContentConfig(tools=[search_courses]))   # automatic function calling is ON
    print(chat.send_message("Which GenAI courses do you have?").text)  # the SDK routes + runs the tool
else:
    print("Set GOOGLE_API_KEY to run this live routing example (illustrative).")

# Output: (illustrative - runs with GOOGLE_API_KEY set)

**Explanation**

Defines a real `search_courses` Python function whose docstring carries the capability + WHEN + negative guidance, hands it to a Gemini chat with automatic function calling ON, and sends one query so the SDK routes to and runs the tool. A guarded API call, not a pure-Python harness.

**How the code works - step by step**
- **`search_courses(topic, max_price_inr)`** - a real function; its docstring is the description the model reads ("Use when the user asks WHICH courses... Do NOT use to enroll or book").
- **key guard** - only runs if `GOOGLE_API_KEY`/`GEMINI_API_KEY` is set, otherwise prints an instruction.
- **`client.chats.create(..., tools=[search_courses])`** - registers the function; automatic function calling means the SDK builds the declaration from type hints + docstring and executes the tool itself.
- **`chat.send_message(...).text`** - the SDK routes to `search_courses`, runs it, and returns the model's text.

**In one line:** the same routing, now driven by a real model reading the docstring-as-description.

**What you'll see:** With a key set, prints the model's reply after it routes to and runs `search_courses`. Without a key, prints "Set GOOGLE_API_KEY to run this live routing example (illustrative)."

## 4 - The portable subset & the strict-mode matrix

Strict mode is a RESTRICTED subset of JSON Schema, and each provider restricts differently - the #1 source of cross-provider tool failures. This cell encodes the compatibility matrix and computes which features survive the strict intersection of the three API providers, revealing how small the portable core really is.

In [ ]:
# THE PORTABLE SUBSET: which JSON Schema features survive each provider's strict mode.
MATRIX = {  # OK / NO / LIMITED / REQ / weak  (from each vendor's docs, Jul 2026)
    "string enum":                  {"OpenAI": "OK",  "Claude": "OK",      "Gemini": "OK",   "MCP": "OK"},
    "additionalProperties:false":   {"OpenAI": "REQ", "Claude": "OK",      "Gemini": "weak", "MCP": "OK"},
    "optional (omit from required)":{"OpenAI": "NO",  "Claude": "OK",      "Gemini": "OK",   "MCP": "OK"},
    "oneOf":                        {"OpenAI": "NO",  "Claude": "LIMITED", "Gemini": "NO",   "MCP": "OK"},
    "anyOf (nested)":               {"OpenAI": "OK",  "Claude": "OK",      "Gemini": "weak", "MCP": "OK"},
    "default keyword":              {"OpenAI": "NO",  "Claude": "NO",      "Gemini": "NO",   "MCP": "OK"},
    "input_examples":               {"OpenAI": "NO",  "Claude": "OK",      "Gemini": "NO",   "MCP": "NO"},
}
def portable(feature):                          # safe on the strict intersection of the 3 providers
    row = MATRIX[feature]
    return all(row[p] in ("OK", "REQ") for p in ("OpenAI", "Claude", "Gemini"))

print(f"{'feature':30s}{'OpenAI':7s}{'Claude':8s}{'Gemini':7s}{'MCP':5s}portable")
for feat, row in MATRIX.items():
    print(f"{feat:30s}{row['OpenAI']:7s}{row['Claude']:8s}{row['Gemini']:7s}{row['MCP']:5s}{portable(feat)}")

# Output:
# feature                       OpenAI Claude  Gemini MCP  portable
# string enum                   OK     OK      OK     OK   True
# additionalProperties:false    REQ    OK      weak   OK   False
# optional (omit from required) NO     OK      OK     OK   False
# oneOf                         NO     LIMITED NO     OK   False
# anyOf (nested)                OK     OK      weak   OK   False
# default keyword               NO     NO      NO     OK   False
# input_examples                NO     OK      NO     NO   False
# -> portable core = flat objects with type/properties/required/enum/description. MCP (full 2020-12) is the outlier.

**Explanation**

A `MATRIX` dict maps each JSON Schema feature to its status per provider, and a `portable` helper marks a feature safe only if all three API providers accept it (`OK`/`REQ`). A reference/measurement harness that reads out a decision table - no model call.

**How the code works - step by step**
- **`MATRIX`** - rows are features (string enum, `additionalProperties:false`, optionals, `oneOf`, `anyOf`, `default`, `input_examples`); columns are OpenAI/Claude/Gemini/MCP with status codes (OK/NO/REQ/LIMITED/weak).
- **`portable(feature)`** - returns True only if OpenAI, Claude, and Gemini all rate it `OK` or `REQ` (MCP is deliberately excluded - it is the permissive outlier).
- **print loop** - renders the matrix as a table with a final `portable` column.

**In one line:** intersect the three strict modes -> the portable core is basically flat objects with enums.

**What you'll see:** A table showing only `string enum` is portable (True); `oneOf`, `default`, `additionalProperties:false`, and `input_examples` all come out non-portable, with a closing note that MCP (full 2020-12) is the outlier that accepts everything.

## 5 - Types, enums & nesting

Three type-level rules move reliability: prefer string enums over boolean pairs (which can contradict), flatten deep object nesting to one or two levels, and don't confuse deep schemas with hard-to-chain call SEQUENCES. This cell quantifies the first two and cites the benchmark for the third.

In [ ]:
# TYPES, ENUMS & NESTING: string enums beat booleans; flatten deep object trees.
def boolean_states():                           # two bools -> 4 combos, 2 of them contradictory
    combos = [(on, off) for on in (True, False) for off in (True, False)]
    invalid = [(on, off) for (on, off) in combos if on == off]   # on==off is a contradiction
    return combos, invalid

def schema_depth(d, level=1):
    if not isinstance(d, dict): return level - 1
    subs = [schema_depth(v, level + 1) for v in d.values() if isinstance(v, dict)]
    return max([level] + subs)

combos, invalid = boolean_states()
print(f"toggle_light(on:bool, off:bool): {len(combos)} states, {len(invalid)} contradictory -> {invalid}")
print("toggle_light(state: 'on'|'off'|'toggle'): 3 states, 0 contradictory  (string enum wins)")

flat   = {"user_city": "Pune", "user_zip": "411001"}                 # depth 1
nested = {"user": {"address": {"city": "Pune", "zip": "411001"}}}    # depth 3
print("flat depth  =", schema_depth(flat), "(recommended 1-2)")
print("nested depth=", schema_depth(nested), "(flatten it)")
print("NESTFUL (IBM, EMNLP 2025): best model ~28% on nested CALL SEQUENCES - a different problem from deep schemas.")

# Output:
# toggle_light(on:bool, off:bool): 4 states, 2 contradictory -> [(True, True), (False, False)]
# toggle_light(state: 'on'|'off'|'toggle'): 3 states, 0 contradictory  (string enum wins)
# flat depth  = 1 (recommended 1-2)
# nested depth= 3 (flatten it)
# NESTFUL (IBM, EMNLP 2025): best model ~28% on nested CALL SEQUENCES - a different problem from deep schemas.

**Explanation**

`boolean_states` enumerates the state space of a two-boolean design to expose its contradictions, and `schema_depth` recursively measures nesting depth of flat vs nested dicts. A measurement harness that makes the "enums win, flatten nesting" advice concrete and countable.

**How the code works - step by step**
- **`boolean_states()`** - builds all four (on, off) combinations and flags the two where `on == off` as contradictory.
- **`schema_depth(d, level)`** - recurses into nested dicts and returns the maximum depth.
- **enum comparison print** - contrasts the 4-state/2-contradictory boolean pair with a single `state` enum (3 states, 0 contradictions).
- **flat vs nested** - measures a depth-1 flat dict against a depth-3 nested one and recommends flattening.
- **NESTFUL note** - reminds you that nested CALL SEQUENCES (~28% best-model score) are a separate, harder problem from deep object schemas.

**In one line:** enums make bad states unrepresentable; flatten schemas; chained calls are a different beast.

**What you'll see:** Prints that the boolean pair has 4 states with 2 contradictory `[(True,True),(False,False)]`, the enum has 3 states/0 contradictory, flat depth = 1 vs nested depth = 3, and the NESTFUL ~28% caveat about call sequences.

## 6 - Granularity: less is more

The counterintuitive finding is that fewer, well-named tools are MORE accurate - past a couple dozen active tools, selection accuracy falls and you should load the long tail dynamically instead. This cell models that accuracy curve and the effect of capping the active set with dynamic tool search.

In [ ]:
# LESS IS MORE: a small active tool set stays accurate; dynamic search caps it.
def selection_accuracy(n_active):
    """Toy model: selection accuracy peaks near a small active set and decays as tools pile up."""
    penalty = 0.006 * max(0, n_active - 15)     # each tool past ~15 chips away at selection
    return round(max(0.35, 0.97 - penalty), 2)
def active_set(total_tools, dynamic_search, cap=15):
    return min(total_tools, cap) if dynamic_search else total_tools

for total, dynamic in [(8, False), (50, False), (50, True)]:
    a = active_set(total, dynamic)
    label = "dynamic tool-search ON" if dynamic else "all tools static"
    print(f"{total:2d} tools, {label:22s} -> active={a:2d}, selection acc ~{selection_accuracy(a)}")
print("Anthropic Tool Search (defer_loading): ~85% context saved; Opus 4.5 79.5% -> 88.1% on large-library MCP evals.")

# Output:
#  8 tools, all tools static       -> active= 8, selection acc ~0.97
# 50 tools, all tools static       -> active=50, selection acc ~0.76
# 50 tools, dynamic tool-search ON -> active=15, selection acc ~0.97
# Anthropic Tool Search (defer_loading): ~85% context saved; Opus 4.5 79.5% -> 88.1% on large-library MCP evals.

**Explanation**

A toy `selection_accuracy` decays as the active tool count climbs past ~15, and `active_set` caps the count when dynamic search is on. A simple simulation that illustrates the shape of the real-world result (with the Anthropic Tool Search numbers cited alongside).

**How the code works - step by step**
- **`selection_accuracy(n_active)`** - starts near 0.97 and subtracts a penalty for every tool beyond 15, floored at 0.35.
- **`active_set(total, dynamic, cap)`** - returns `min(total, cap)` when dynamic search is on, else the full count.
- **scenario loop** - runs 8-static, 50-static, and 50-with-dynamic-search and prints the resulting active count and accuracy.
- **citation line** - Anthropic's Tool Search (`defer_loading`) saves ~85% context and lifted Opus 4.5 from 79.5% to 88.1% on large-library MCP evals.

**In one line:** keep a small active set; defer-load the rest to recover accuracy.

**What you'll see:** Prints ~0.97 for 8 static tools, a drop to ~0.76 for 50 static tools, and recovery to ~0.97 when dynamic search caps the active set at 15, plus the Anthropic Tool Search numbers.

## 7 - Side effects & safety

A schema declares what a tool DOES to the world, so two rules keep agents safe: every write tool takes an idempotency_key (retries must not double-charge), and side effects are classified with MCP annotations that map to human-approval risk tiers. This cell demonstrates both mechanisms.

In [ ]:
# SIDE EFFECTS: idempotency keys stop double-charges; MCP annotations set the approval tier.
processed = {}
def charge(idempotency_key, amount_inr):
    if idempotency_key in processed:                        # agent retried the SAME key
        return {"status": "duplicate ignored", "amount": processed[idempotency_key]}
    processed[idempotency_key] = amount_inr
    return {"status": "charged", "amount": amount_inr}

def risk_tier(annotations):                                 # MCP tool annotations -> human-in-the-loop tier
    if annotations.get("readOnlyHint"):            return "C (auto-approve): read-only"
    if annotations.get("destructiveHint", True):   return "A (human approval): irreversible"
    return "B (recommended approval): reversible write"

print("agent retries a payment with the SAME idempotency_key:")
print(" ", charge("pay-8821", 14999))          # first call charges
print(" ", charge("pay-8821", 14999))          # retry is ignored -> no double charge
for name, ann in [("get_weather", {"readOnlyHint": True}),
                  ("update_profile", {"readOnlyHint": False, "destructiveHint": False, "idempotentHint": True}),
                  ("delete_account", {"readOnlyHint": False, "destructiveHint": True})]:
    print(f"  {name:15s} -> {risk_tier(ann)}")

# Output:
# agent retries a payment with the SAME idempotency_key:
#   {'status': 'charged', 'amount': 14999}
#   {'status': 'duplicate ignored', 'amount': 14999}
#   get_weather     -> C (auto-approve): read-only
#   update_profile  -> B (recommended approval): reversible write
#   delete_account  -> A (human approval): irreversible

**Explanation**

`charge` records processed idempotency keys and short-circuits duplicates, while `risk_tier` reads MCP annotation hints to assign an approval tier. A small stateful harness that shows how the schema itself encodes safety and human-in-the-loop gating.

**How the code works - step by step**
- **`charge(idempotency_key, amount_inr)`** - if the key was already processed, returns `duplicate ignored`; otherwise records it and charges - so a retry with the same key cannot double-charge.
- **`risk_tier(annotations)`** - maps `readOnlyHint` -> tier C (auto), `destructiveHint` (default True) -> tier A (human approval), else tier B (reversible write, recommended approval).
- **retry demo** - calls `charge("pay-8821", ...)` twice with the same key.
- **annotation loop** - classifies `get_weather`, `update_profile`, and `delete_account` into tiers C/B/A.

**In one line:** idempotency keys stop double-acts; annotations gate irreversible calls behind approval.

**What you'll see:** Prints the first charge succeeding (`charged`), the identical retry returning `duplicate ignored` (no second charge), and the three tools sorted into tier C (read-only), tier B (reversible write), and tier A (irreversible / human approval).

## 8 - Versioning & evaluation

A tool schema is a production contract that can break agents silently - no error, just wrong behavior - so you semver it, know which changes are breaking, and gate CI on a golden test suite. This cell classifies breaking changes and scores a set of versions against a pass threshold.

In [ ]:
# VERSIONING & EVAL: classify breaking changes; gate CI on a golden test suite.
def is_breaking(change, strict_mode):
    hard = {"rename param", "change type", "optional->required", "remove param"}
    if change in hard:       return True
    if change == "add field": return strict_mode    # strict additionalProperties:false -> adding a field breaks
    return False                                     # improve description / add enum value = safe

def tool_correctness(expected, actual):              # DeepEval-style: correct / total called
    total = len(actual) or 1
    return round(len(set(expected) & set(actual)) / total, 2)

def golden_gate(scores, threshold=0.90):
    return "PASS" if min(scores) >= threshold else "BLOCK deploy (regression)"

for ch in ["add field", "improve description", "rename param", "optional->required"]:
    print(f"{ch:22s} strict={is_breaking(ch, True)}  lenient={is_breaking(ch, False)}")
print("ToolCorrectnessMetric:", tool_correctness(["search_courses"], ["search_courses"]),
      "vs", tool_correctness(["search_courses"], ["get_weather", "search_courses"]))
print("golden gate over 3 versions:", golden_gate([0.94, 0.91, 0.86]), "(a 5% drop is a regression)")

# Output:
# add field              strict=True  lenient=False
# improve description    strict=False  lenient=False
# rename param           strict=True  lenient=True
# optional->required     strict=True  lenient=True
# ToolCorrectnessMetric: 1.0 vs 0.5
# golden gate over 3 versions: BLOCK deploy (regression) (a 5% drop is a regression)

**Explanation**

`is_breaking` classifies schema changes (with `add field` breaking only under strict mode), `tool_correctness` scores correct-vs-total tool calls DeepEval-style, and `golden_gate` blocks a deploy if any version dips below threshold. A decision harness that treats the schema as a versioned API contract.

**How the code works - step by step**
- **`is_breaking(change, strict_mode)`** - renames/type-changes/optional->required/remove are always breaking; `add field` is breaking only when `strict_mode` (because `additionalProperties:false`); description tweaks are safe.
- **`tool_correctness(expected, actual)`** - correct tools over total called (an extra wrong tool drags the score down).
- **`golden_gate(scores, threshold)`** - PASS only if the minimum score clears 0.90, else BLOCK.
- **print block** - classifies four changes under strict vs lenient, scores a clean vs noisy call, and gates three versions where one scores 0.86.

**In one line:** semver the schema, know strict-mode adds are breaking, and let golden tests block regressions.

**What you'll see:** Prints that `add field` is `strict=True / lenient=False`, `rename param` and `optional->required` break in both modes, description tweaks are safe, ToolCorrectness of `1.0` vs `0.5`, and the golden gate returning `BLOCK deploy (regression)` because one version scored 0.86.

## 9 - India schema design

Indic apps add correctness requirements a demo never surfaces: normalize text to Unicode NFC before comparing, count graphemes not code points for length limits, and validate Indian identifiers with real patterns. This cell demonstrates all three plus the identifier regexes.

In [ ]:
# INDIA: Unicode NFC + grapheme-aware length + Indian identifier validation.
import unicodedata, re
precomposed = "क़"          # a single code point (nukta form)
decomposed  = "क़"    # base + nukta (two code points)
print("precomposed == decomposed?", precomposed == decomposed,
      "-> after NFC:", unicodedata.normalize("NFC", precomposed) == unicodedata.normalize("NFC", decomposed))

ksha = "क्ष"     # one conjunct built from 3 code points, 1 visible grapheme
print(f"conjunct code points = {len(ksha)} but 1 grapheme -> maxLength (code points) overcounts; use grapheme-aware checks")

PATTERNS = {"PAN": r"^[A-Z]{5}[0-9]{4}[A-Z]$",
            "GSTIN": r"^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][0-9A-Z]Z[0-9A-Z]$",
            "mobile": r"^\+91[6-9][0-9]{9}$",
            "pincode": r"^[1-9][0-9]{5}$"}
samples = {"PAN": "ABCDE1234F", "GSTIN": "27ABCDE1234F1Z5", "mobile": "+919876543210", "pincode": "411001"}
for key, pat in PATTERNS.items():
    print(f"  {key:8s} {samples[key]:16s} -> {'valid' if re.match(pat, samples[key]) else 'INVALID'}")

# Output:
# precomposed == decomposed? False -> after NFC: True
# conjunct code points = 3 but 1 grapheme -> maxLength (code points) overcounts; use grapheme-aware checks
#   PAN      ABCDE1234F       -> valid
#   GSTIN    27ABCDE1234F1Z5  -> valid
#   mobile   +919876543210    -> valid
#   pincode  411001           -> valid

**Explanation**

The cell shows two visually identical nukta forms comparing unequal until NFC-normalized, measures a conjunct that is 3 code points but 1 grapheme, and regex-validates PAN/GSTIN/mobile/pincode samples. A validation harness for the Indic-correctness rules that separate a real app from a demo.

**How the code works - step by step**
- **NFC block** - a precomposed vs base+nukta string compares `False` as raw code points but `True` after `unicodedata.normalize("NFC", ...)` - always normalize before compare/store.
- **grapheme block** - `len("क्ष")` is 3 code points for 1 visible grapheme, so a code-point `maxLength` overcounts; use grapheme-aware checks.
- **`PATTERNS` + `samples`** - regexes for PAN, GSTIN, mobile, and pincode, each tested against a valid sample.
- **validation loop** - prints valid/INVALID per identifier - catch malformed IDs at the schema boundary, not the database.

**In one line:** NFC-normalize, count graphemes, and regex-validate Indian identifiers at the schema edge.

**What you'll see:** Prints `precomposed == decomposed? False -> after NFC: True`, that the conjunct is 3 code points / 1 grapheme, and each of PAN/GSTIN/mobile/pincode validating as `valid`.

Across eight small harnesses you saw the same idea from every angle: a tool schema is a prompt and a contract, not just a shape. One Pydantic model feeds every provider; the description drives routing; the portable subset survives all strict modes; string enums, flat nesting, small tool sets, idempotency keys, semver + golden tests, and NFC/identifier validation are the levers that separate a schema that demos from one that survives production. Next, Lesson 10.3 orchestrates multiple tools together (chaining, parallel calls, agent loops) - where these well-designed schemas pay off most - and Lessons 10.4/10.5 build the MCP server that exposes them as `inputSchema`.